# COMPLETE MACHINE LEARNING JOURNEY: Baseline → Advanced CNN
  Dataset: MNIST Handwritten Digits (loaded via TensorFlow/Keras — no manual download)
  Libraries: NumPy, Pandas, Matplotlib, Seaborn, Scikit-Learn, TensorFlow/Keras

TABLE OF CONTENTS
-----------------
 1. Setup & Imports
 2. Data Loading & Exploration (EDA)
 3. Data Preprocessing
 4. BASELINE MODEL — Logistic Regression
 5. IMPROVEMENT 1 — Decision Tree
 6. IMPROVEMENT 2 — Random Forest
 7. IMPROVEMENT 3 — Support Vector Machine (SVM)
 8. IMPROVEMENT 4 — Basic Neural Network (MLP via Keras)
 9. IMPROVEMENT 5 — Deeper MLP with Batch Normalization & Dropout
10. IMPROVEMENT 6 — Convolutional Neural Network (CNN)
11. IMPROVEMENT 7 — Advanced CNN with Data Augmentation
12. Final Comparison Dashboard

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe for scripts)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
import time
import os

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
# Scikit-Learn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA

In [ ]:
# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
print("=" * 70)
print("  ML JOURNEY: MNIST Handwritten Digit Classification")
print("=" * 70)
print(f"\n  TensorFlow version : {tf.__version__}")
print(f"  NumPy version      : {np.__version__}")
print(f"  Pandas version     : {pd.__version__}")
 

In [ ]:
OUTPUT_DIR = "ml_journey_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Palette used throughout
PALETTE     = "viridis"
ACCENT      = "#4C72B0"
SUCCESS_CLR = "#2ecc71"
DANGER_CLR  = "#e74c3c"

# SECTION 2: DATA LOADING & EXPLORATORY DATA ANALYSIS

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 2 — Data Loading & Exploratory Data Analysis")
print("=" * 70)

In [ ]:
#  Load MNIST 
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()
 
print(f"\n  Raw training set  : {X_train_raw.shape}  labels: {y_train_raw.shape}")
print(f"  Raw test set      : {X_test_raw.shape}  labels: {y_test_raw.shape}")
 

In [ ]:
# Convert to DataFrame for Pandas EDA
n_pixels = 28 * 28
X_train_flat_raw = X_train_raw.reshape(-1, n_pixels)
X_test_flat_raw  = X_test_raw.reshape(-1, n_pixels)
 
col_names  = [f"pixel_{i}" for i in range(n_pixels)]
df_train   = pd.DataFrame(X_train_flat_raw, columns=col_names)
df_train["label"] = y_train_raw

print("\n  ── Pandas DataFrame (first 5 rows, first 10 pixel cols + label) ──")
print(df_train[col_names[:10] + ["label"]].head())
 
print("\n  ── Basic Statistics (pixel_0 … pixel_9) ──")
print(df_train[col_names[:10]].describe().round(2))
 
print("\n  ── Class Distribution ──")
class_dist = df_train["label"].value_counts().sort_index()
print(class_dist.to_string())

In [ ]:
# EDA Figure 1: Sample images + class distribution

fig = plt.figure(figsize=(20, 10))
fig.patch.set_facecolor("#1a1a2e")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

In [ ]:
#  Sample images
ax_imgs = fig.add_subplot(gs[0, 0])
ax_imgs.set_facecolor("#1a1a2e")
ax_imgs.axis("off")
ax_imgs.set_title("Sample Images (one per class)", color="white", fontsize=14, pad=10)
for digit in range(10):
    idx = np.where(y_train_raw == digit)[0][0]
    ax_inner = fig.add_axes([0.03 + digit * 0.047, 0.55, 0.042, 0.1])
    ax_inner.imshow(X_train_raw[idx], cmap="plasma")
    ax_inner.set_title(str(digit), color="white", fontsize=9)
    ax_inner.axis("off")
 

In [ ]:
#  Class distribution bar
ax_bar = fig.add_subplot(gs[0, 1])
ax_bar.set_facecolor("#16213e")
colors = sns.color_palette("plasma", 10)
bars   = ax_bar.bar(class_dist.index, class_dist.values, color=colors, edgecolor="white", linewidth=0.5)
ax_bar.set_title("Class Distribution (Training Set)", color="white", fontsize=13)
ax_bar.set_xlabel("Digit", color="#aaaaaa")
ax_bar.set_ylabel("Count", color="#aaaaaa")
ax_bar.tick_params(colors="white")
for spine in ax_bar.spines.values():
    spine.set_edgecolor("#444466")
for bar, cnt in zip(bars, class_dist.values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                f"{cnt:,}", ha="center", va="bottom", color="white", fontsize=8)
 

In [ ]:
# Pixel intensity histogram
ax_hist = fig.add_subplot(gs[1, 0])
ax_hist.set_facecolor("#16213e")
sample_pixels = X_train_flat_raw[:5000].flatten()
ax_hist.hist(sample_pixels, bins=50, color=ACCENT, edgecolor="none", alpha=0.85)
ax_hist.set_title("Pixel Intensity Distribution (5k samples)", color="white", fontsize=13)
ax_hist.set_xlabel("Pixel Value (0–255)", color="#aaaaaa")
ax_hist.set_ylabel("Frequency", color="#aaaaaa")
ax_hist.tick_params(colors="white")
for spine in ax_hist.spines.values():
    spine.set_edgecolor("#444466")

In [ ]:
#  Mean digit images
ax_mean = fig.add_subplot(gs[1, 1])
ax_mean.axis("off")
ax_mean.set_facecolor("#1a1a2e")
ax_mean.set_title("Mean Image per Class", color="white", fontsize=13, pad=10)
for digit in range(10):
    mask       = y_train_raw == digit
    mean_img   = X_train_raw[mask].mean(axis=0)
    ax_m = fig.add_axes([0.54 + (digit % 5) * 0.087,
                         0.08 + (1 - digit // 5) * 0.19,
                         0.075, 0.15])
    ax_m.imshow(mean_img, cmap="inferno")
    ax_m.set_title(f"μ({digit})", color="white", fontsize=8)
    ax_m.axis("off")
 
fig.suptitle("MNIST — Exploratory Data Analysis", color="white",
             fontsize=18, fontweight="bold", y=0.98)
path_eda = os.path.join(OUTPUT_DIR, "01_eda.png")
plt.savefig(path_eda, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"\n  [✓] EDA plot saved → {path_eda}")
 

In [ ]:
# EDA Figure 2: Correlation heatmap on PCA-reduced features
print("\n  Computing PCA (20 components) for correlation heatmap …")
pca_eda   = PCA(n_components=20, random_state=SEED)
X_pca_eda = pca_eda.fit_transform(X_train_flat_raw[:10000] / 255.0)
df_pca    = pd.DataFrame(X_pca_eda, columns=[f"PC{i+1}" for i in range(20)])
df_pca["label"] = y_train_raw[:10000]
 
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor("#1a1a2e")

ax1 = axes[0]
ax1.set_facecolor("#16213e")
corr = df_pca.drop("label", axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0,
            annot=False, linewidths=0.3, ax=ax1,
            cbar_kws={"shrink": 0.8})
ax1.set_title("PCA Component Correlation Matrix", color="white", fontsize=13, pad=10)
ax1.tick_params(colors="white", labelsize=8)


ax2 = axes[1]
ax2.set_facecolor("#16213e")
explained = pca_eda.explained_variance_ratio_ * 100
cumulative = np.cumsum(explained)
ax2.bar(range(1, 21), explained, color=sns.color_palette("plasma", 20), label="Individual")
ax2.plot(range(1, 21), cumulative, "w-o", linewidth=2, markersize=5, label="Cumulative")
ax2.axhline(80, color=SUCCESS_CLR, linestyle="--", linewidth=1.5, label="80% threshold")
ax2.set_title("PCA Explained Variance", color="white", fontsize=13)
ax2.set_xlabel("Principal Component", color="#aaaaaa")
ax2.set_ylabel("Variance Explained (%)", color="#aaaaaa")
ax2.tick_params(colors="white")
legend = ax2.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
 
for spine in ax2.spines.values():
    spine.set_edgecolor("#444466")
 
fig.suptitle("MNIST — PCA Analysis", color="white", fontsize=16, fontweight="bold")
path_pca = os.path.join(OUTPUT_DIR, "02_pca_analysis.png")
plt.savefig(path_pca, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] PCA analysis plot saved → {path_pca}")


# SECTION 3: DATA PREPROCESSING


In [ ]:
print("\n" + "=" * 70)
print("  SECTION 3 — Data Preprocessing")
print("=" * 70)


In [ ]:
# Normalize pixel values to [0, 1] 
X_train_norm = X_train_flat_raw / 255.0    # shape: (60000, 784)
X_test_norm  = X_test_flat_raw  / 255.0    # shape: (10000, 784)

In [ ]:
# 2-D versions for CNN 
X_train_2d = X_train_raw.reshape(-1, 28, 28, 1) / 255.0
X_test_2d  = X_test_raw.reshape(-1, 28, 28, 1)  / 255.0
 


In [ ]:
# One-hot labels for Keras 
y_train_oh = keras.utils.to_categorical(y_train_raw, 10)
y_test_oh  = keras.utils.to_categorical(y_test_raw,  10)

In [ ]:
# Subsample for slower sklearn models
SKLEARN_TRAIN = 20_000
SKLEARN_TEST  = 5_000
 
X_sk_train = X_train_norm[:SKLEARN_TRAIN]
y_sk_train = y_train_raw[:SKLEARN_TRAIN]
X_sk_test  = X_test_norm[:SKLEARN_TEST]
y_sk_test  = y_test_raw[:SKLEARN_TEST]

print(f"\n  Normalized pixel range : [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")
print(f"  Sklearn training size  : {SKLEARN_TRAIN:,}  test size: {SKLEARN_TEST:,}")
print(f"  Keras training size    : {X_train_norm.shape[0]:,}  test size: {X_test_norm.shape[0]:,}")

In [ ]:
# Storage for final comparison
results = {}

In [ ]:
#  HELPER FUNCTIONS
def evaluate_sklearn(name, model, X_tr, y_tr, X_te, y_te):
    """Train, time, evaluate, and store a scikit-learn model."""
    print(f"\n  Training {name} …")
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
 
    y_pred = model.predict(X_te)
    acc    = accuracy_score(y_te, y_pred)
    report = classification_report(y_te, y_pred, output_dict=True)
    cm     = confusion_matrix(y_te, y_pred)
 
    results[name] = {"accuracy": acc, "train_time": train_time,
                     "history": None, "type": "sklearn"}
    print(f"  {name:40s} | Accuracy: {acc*100:.2f}% | Time: {train_time:.1f}s")
    return y_pred, cm, report

In [ ]:
def plot_confusion_matrix(cm, title, filepath, cmap="Blues"):
    """Styled confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#16213e")
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap=cmap,
                linewidths=0.4, linecolor="#2a2a4a",
                xticklabels=range(10), yticklabels=range(10),
                cbar_kws={"label": "% of True Class"}, ax=ax)
    ax.set_title(title, color="white", fontsize=14, pad=12)
    ax.set_xlabel("Predicted Label", color="#aaaaaa")
    ax.set_ylabel("True Label", color="#aaaaaa")
    ax.tick_params(colors="white")
    plt.colorbar(ax.collections[0], ax=ax).ax.yaxis.label.set_color("white")
    plt.tight_layout()
    plt.savefig(filepath, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()

In [ ]:
def plot_keras_history(history, title, filepath):
    """Two-panel loss + accuracy plot for Keras training."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#1a1a2e")
 
    for ax, metric, ylabel in zip(
        axes, ["loss", "accuracy"], ["Loss", "Accuracy"]
    ):
        ax.set_facecolor("#16213e")
        ax.plot(history.history[metric],
                color=ACCENT, linewidth=2.5, label="Train")
        ax.plot(history.history[f"val_{metric}"],
                color=SUCCESS_CLR, linewidth=2.5,
                linestyle="--", label="Validation")
        ax.set_title(f"{title} — {ylabel}", color="white", fontsize=13)
        ax.set_xlabel("Epoch", color="#aaaaaa")
        ax.set_ylabel(ylabel, color="#aaaaaa")
        ax.tick_params(colors="white")
        legend = ax.legend(facecolor="#2a2a4a",
                           edgecolor="white", labelcolor="white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444466")
 
    plt.tight_layout()
    plt.savefig(filepath, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
 

In [ ]:
def show_misclassified(X_flat, y_true, y_pred, title, filepath, n=20):
    """Grid of misclassified examples."""
    wrong_idx = np.where(y_true != y_pred)[0][:n]
    cols, rows = 10, 2
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(title, color="white", fontsize=14, y=1.01)
    for i, idx in enumerate(wrong_idx):
        ax = axes[i // cols][i % cols]
        ax.imshow(X_flat[idx].reshape(28, 28), cmap="plasma")
        ax.set_title(f"T:{y_true[idx]} P:{y_pred[idx]}",
                     color=DANGER_CLR, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(filepath, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()

# SECTION 4: BASELINE — LOGISTIC REGRESSION

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 4 — Baseline: Logistic Regression")
print("=" * 70)
 
lr_model = LogisticRegression(
    max_iter=1000, solver="lbfgs", multi_class="multinomial",
    C=1.0, random_state=SEED
)
lr_pred, lr_cm, lr_report = evaluate_sklearn(
    "Logistic Regression", lr_model,
    X_sk_train, y_sk_train, X_sk_test, y_sk_test
)
plot_confusion_matrix(
    lr_cm, "Logistic Regression — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "03_lr_confusion.png"), cmap="Blues"
)
show_misclassified(
    X_sk_test, y_sk_test, lr_pred,
    "Logistic Regression — Misclassified Examples",
    os.path.join(OUTPUT_DIR, "04_lr_misclassified.png")
)

# Per-class bar chart
df_lr = pd.DataFrame(lr_report).T
df_lr = df_lr[df_lr.index.isin([str(i) for i in range(10)])]
 
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
x    = np.arange(10)
w    = 0.25
bars = [
    ax.bar(x - w,     df_lr["precision"].astype(float), w, label="Precision", color="#4C72B0"),
    ax.bar(x,         df_lr["recall"].astype(float),    w, label="Recall",    color="#DD8452"),
    ax.bar(x + w,     df_lr["f1-score"].astype(float),  w, label="F1-Score",  color="#55A868"),
]
ax.set_xticks(x)
ax.set_xticklabels(range(10), color="white")
ax.set_title("Logistic Regression — Per-Class Metrics", color="white", fontsize=13)
ax.set_xlabel("Digit", color="#aaaaaa")
ax.set_ylabel("Score", color="#aaaaaa")
ax.tick_params(colors="white")
ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
ax.set_ylim(0, 1.1)
for spine in ax.spines.values():
    spine.set_edgecolor("#444466")
path_lr_metrics = os.path.join(OUTPUT_DIR, "05_lr_per_class.png")
plt.tight_layout()
plt.savefig(path_lr_metrics, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] LR per-class metrics saved → {path_lr_metrics}")
 





# SECTION 5: IMPROVEMENT 1 — DECISION TREE


In [ ]:
print("\n" + "=" * 70)
print("  SECTION 5 — Improvement 1: Decision Tree")
print("=" * 70)
 
dt_model = DecisionTreeClassifier(
    max_depth=20, min_samples_split=5,
    criterion="gini", random_state=SEED
)

In [ ]:
dt_pred, dt_cm, _ = evaluate_sklearn(
    "Decision Tree (max_depth=20)", dt_model,
    X_sk_train, y_sk_train, X_sk_test, y_sk_test
)

In [ ]:
plot_confusion_matrix(
    dt_cm, "Decision Tree — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "06_dt_confusion.png"), cmap="Purples"
)

In [ ]:
# Depth vs accuracy plot
print("  Tuning Decision Tree max_depth …")
depths = [3, 5, 8, 12, 15, 20, 25, 30, None]
accs   = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    m.fit(X_sk_train, y_sk_train)
    accs.append(accuracy_score(y_sk_test, m.predict(X_sk_test)))
    print(f"    depth={str(d):4s} → {accs[-1]*100:.2f}%")
 
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
depth_labels = [str(d) if d else "None" for d in depths]
ax.plot(depth_labels, [a * 100 for a in accs], "o-",
        color=ACCENT, linewidth=2.5, markersize=9)
ax.fill_between(depth_labels, [a * 100 for a in accs],
                alpha=0.15, color=ACCENT)
ax.set_title("Decision Tree — Depth vs Accuracy", color="white", fontsize=13)
ax.set_xlabel("max_depth", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444466")
plt.tight_layout()
path_dt_depth = os.path.join(OUTPUT_DIR, "07_dt_depth_tuning.png")
plt.savefig(path_dt_depth, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] DT depth-tuning saved → {path_dt_depth}")

SECTION 6: IMPROVEMENT 2 — RANDOM FOREST

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 6 — Improvement 2: Random Forest")
print("=" * 70)
 
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=None,
    min_samples_split=2, n_jobs=-1, random_state=SEED
)
rf_pred, rf_cm, rf_report = evaluate_sklearn(
    "Random Forest (200 trees)", rf_model,
    X_sk_train, y_sk_train, X_sk_test, y_sk_test
)
plot_confusion_matrix(
    rf_cm, "Random Forest — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "08_rf_confusion.png"), cmap="Greens"
)

In [ ]:
# Feature importance (top 30 pixels)
importances  = rf_model.feature_importances_
imp_img      = importances.reshape(28, 28)
fig, axes    = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#1a1a2e")
axes[0].set_facecolor("#16213e")
im = axes[0].imshow(imp_img, cmap="hot")


axes[0].set_title("RF Feature Importances (pixel space)",
                  color="white", fontsize=12)
axes[0].axis("off")
plt.colorbar(im, ax=axes[0]).ax.yaxis.label.set_color("white")
 
top30_idx   = np.argsort(importances)[::-1][:30]
top30_vals  = importances[top30_idx]

axes[1].set_facecolor("#16213e")
axes[1].barh(range(30), top30_vals[::-1],
             color=sns.color_palette("plasma", 30))
axes[1].set_yticks(range(30))
axes[1].set_yticklabels([f"px_{i}" for i in top30_idx[::-1]],
                         fontsize=7, color="white")
axes[1].set_title("Top 30 Most Important Pixels", color="white", fontsize=12)
axes[1].set_xlabel("Importance", color="#aaaaaa")
axes[1].tick_params(colors="white")

for spine in axes[1].spines.values():
    spine.set_edgecolor("#444466")
plt.tight_layout()
path_rf_fi = os.path.join(OUTPUT_DIR, "09_rf_feature_importance.png")
plt.savefig(path_rf_fi, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] RF feature importance saved → {path_rf_fi}")


# SECTION 7: IMPROVEMENT 3 — SUPPORT VECTOR MACHINE

In [ ]:

print("\n" + "=" * 70)
print("  SECTION 7 — Improvement 3: Support Vector Machine (SVM)")
print("=" * 70)
 
# Use a smaller subset for SVM (O(n²) training complexity)
SVM_SIZE = 10_000
svm_model = SVC(
    kernel="rbf", C=10.0, gamma="scale",
    decision_function_shape="ovr", random_state=SEED
)
svm_pred, svm_cm, _ = evaluate_sklearn(
    "SVM — RBF kernel (C=10)", svm_model,
    X_sk_train[:SVM_SIZE], y_sk_train[:SVM_SIZE],
    X_sk_test, y_sk_test
)

# Update results key with correct name
results["SVM — RBF kernel (C=10)"] = results.pop("SVM — RBF kernel (C=10)")

plot_confusion_matrix(
    svm_cm, "SVM RBF — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "10_svm_confusion.png"), cmap="YlOrRd"
)

In [ ]:
# C parameter sensitivity

print("  Tuning SVM C parameter …")
C_values = [0.1, 1, 5, 10, 50]
svm_accs = []
for c in C_values:
    m = SVC(kernel="rbf", C=c, gamma="scale", random_state=SEED)
    m.fit(X_sk_train[:SVM_SIZE], y_sk_train[:SVM_SIZE])
    acc = accuracy_score(y_sk_test, m.predict(X_sk_test))
    svm_accs.append(acc)
    print(f"    C={c:5} → {acc*100:.2f}%")
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
ax.semilogx(C_values, [a * 100 for a in svm_accs], "o-",
            color="#e74c3c", linewidth=2.5, markersize=9)
ax.fill_between(C_values, [a * 100 for a in svm_accs],
                alpha=0.15, color="#e74c3c")
ax.set_title("SVM RBF — C Parameter Sensitivity", color="white", fontsize=13)
ax.set_xlabel("C (log scale)", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444466")
plt.tight_layout()
path_svm_c = os.path.join(OUTPUT_DIR, "11_svm_C_sensitivity.png")
plt.savefig(path_svm_c, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] SVM C sensitivity saved → {path_svm_c}")


# SECTION 8: IMPROVEMENT 4 — BASIC MLP NEURAL NETWORK (KERAS)

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 8 — Improvement 4: Basic MLP Neural Network")
print("=" * 70)
 
def build_basic_mlp():
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10,  activation="softmax"),
    ], name="Basic_MLP")
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
mlp_basic = build_basic_mlp()
mlp_basic.summary()
 
t0 = time.time()
hist_mlp_basic = mlp_basic.fit(
    X_train_norm, y_train_oh,
    epochs=30, batch_size=256,
    validation_split=0.1,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)
mlp_basic_time = time.time() - t0
mlp_basic_acc  = mlp_basic.evaluate(X_test_norm, y_test_oh, verbose=0)[1]


results["Basic MLP"] = {
    "accuracy": mlp_basic_acc, "train_time": mlp_basic_time,
    "history": hist_mlp_basic.history, "type": "keras"
}
print(f"\n  Basic MLP Test Accuracy: {mlp_basic_acc*100:.2f}%")



In [ ]:
plot_keras_history(
    hist_mlp_basic, "Basic MLP",
    os.path.join(OUTPUT_DIR, "12_basic_mlp_history.png")
)

In [ ]:
mlp_basic_pred = np.argmax(mlp_basic.predict(X_test_norm, verbose=0), axis=1)
plot_confusion_matrix(
    confusion_matrix(y_test_raw, mlp_basic_pred),
    "Basic MLP — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "13_basic_mlp_confusion.png"), cmap="Blues"
)

# SECTION 9: IMPROVEMENT 5 — DEEPER MLP WITH BN + DROPOUT

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 9 — Improvement 5: Deep MLP (BatchNorm + Dropout)")
print("=" * 70)
 
def build_deep_mlp():
    inp = layers.Input(shape=(784,))
    x   = layers.Dense(512, activation="relu")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.35)(x)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.30)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.20)(x)
    x   = layers.Dense(64,  activation="relu")(x)
    out = layers.Dense(10,  activation="softmax")(x)
    model = keras.Model(inp, out, name="Deep_MLP_BN_Dropout")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
mlp_deep = build_deep_mlp()
mlp_deep.summary()

In [ ]:
callbacks_deep = [
    EarlyStopping(patience=7, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

In [ ]:
t0 = time.time()
hist_mlp_deep = mlp_deep.fit(
    X_train_norm, y_train_oh,
    epochs=60, batch_size=256,
    validation_split=0.1,
    callbacks=callbacks_deep,
    verbose=1
)

mlp_deep_time = time.time() - t0
mlp_deep_acc  = mlp_deep.evaluate(X_test_norm, y_test_oh, verbose=0)[1]
results["Deep MLP (BN+Dropout)"] = {
    "accuracy": mlp_deep_acc, "train_time": mlp_deep_time,
    "history": hist_mlp_deep.history, "type": "keras"
}
print(f"\n  Deep MLP Test Accuracy: {mlp_deep_acc*100:.2f}%")

In [ ]:

 
plot_keras_history(
    hist_mlp_deep, "Deep MLP (BN+Dropout)",
    os.path.join(OUTPUT_DIR, "14_deep_mlp_history.png")
)

In [ ]:
mlp_deep_pred = np.argmax(mlp_deep.predict(X_test_norm, verbose=0), axis=1)
plot_confusion_matrix(
    confusion_matrix(y_test_raw, mlp_deep_pred),
    "Deep MLP — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "15_deep_mlp_confusion.png"), cmap="Purples"
)


In [ ]:
# Visualise weight distributions for layer 1 and layer 3
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor("#1a1a2e")
layer_names = ["Dense_512", "Dense_256", "Dense_128"]
dense_layers = [l for l in mlp_deep.layers if isinstance(l, layers.Dense)][:3]
for ax, layer, name in zip(axes, dense_layers, layer_names):
    ax.set_facecolor("#16213e")
    weights = layer.get_weights()[0].flatten()
    ax.hist(weights, bins=60, color=ACCENT, edgecolor="none", alpha=0.8)
    ax.set_title(f"{name} — Weight Distribution", color="white", fontsize=11)
    ax.set_xlabel("Weight Value", color="#aaaaaa")
    ax.set_ylabel("Frequency", color="#aaaaaa")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444466")
fig.suptitle("Deep MLP — Weight Distributions After Training",
             color="white", fontsize=14)
plt.tight_layout()
path_weights = os.path.join(OUTPUT_DIR, "16_deep_mlp_weights.png")
plt.savefig(path_weights, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] Weight distributions saved → {path_weights}")
 


# SECTION 10: IMPROVEMENT 6 — ç

In [ ]:
print("\n" + "=" * 70)
print("  SECTION 10 — Improvement 6: Convolutional Neural Network (CNN)")
print("=" * 70)
 
def build_cnn():
    inp = layers.Input(shape=(28, 28, 1))
 
    # Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(0.30)(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(0.35)(x)

    # Classifier head
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.Dropout(0.40)(x)
    out = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inp, out, name="CNN")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn = build_cnn()
cnn.summary()
 
callbacks_cnn = [
    EarlyStopping(patience=8, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.4, patience=3, min_lr=1e-7, verbose=1),
]
t0 = time.time()
hist_cnn = cnn.fit(
    X_train_2d, y_train_oh,
    epochs=50, batch_size=128,
    validation_split=0.1,
    callbacks=callbacks_cnn,
    verbose=1
)
 

In [ ]:
cnn_time = time.time() - t0
cnn_acc  = cnn.evaluate(X_test_2d, y_test_oh, verbose=0)[1]
results["CNN (3 conv blocks)"] = {
    "accuracy": cnn_acc, "train_time": cnn_time,
    "history": hist_cnn.history, "type": "keras"
}
print(f"\n  CNN Test Accuracy: {cnn_acc*100:.2f}%")
 

In [ ]:
plot_keras_history(
    hist_cnn, "CNN (3 Conv Blocks)",
    os.path.join(OUTPUT_DIR, "17_cnn_history.png")
)

In [ ]:
cnn_pred = np.argmax(cnn.predict(X_test_2d, verbose=0), axis=1)
plot_confusion_matrix(
    confusion_matrix(y_test_raw, cnn_pred),
    "CNN — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "18_cnn_confusion.png"), cmap="plasma"
)

In [ ]:
# Visualise learned filters (Conv layer 1)
conv1_weights = cnn.layers[1].get_weights()[0]   # shape: (3,3,1,32)
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.patch.set_facecolor("#1a1a2e")
fig.suptitle("CNN — Learned Filters (Conv Block 1)", color="white", fontsize=14)
for i, ax in enumerate(axes.flat):
    if i < 32:
        filt = conv1_weights[:, :, 0, i]
        ax.imshow(filt, cmap="RdBu_r", vmin=filt.min(), vmax=filt.max())
    ax.axis("off")
plt.tight_layout()
path_filters = os.path.join(OUTPUT_DIR, "19_cnn_filters.png")
plt.savefig(path_filters, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] CNN filters saved → {path_filters}")
 

In [ ]:
# Feature maps for one test image
sample_idx = 7
sample_img = X_test_2d[sample_idx:sample_idx+1]
# Build feature extractor to first conv block output
feat_model = keras.Model(inputs=cnn.input, outputs=cnn.layers[3].output)
feat_maps  = feat_model.predict(sample_img, verbose=0)[0]  # (14,14,32)
 
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.patch.set_facecolor("#1a1a2e")
fig.suptitle(f"CNN — Feature Maps (after Conv Block 1) — True label: {y_test_raw[sample_idx]}",
             color="white", fontsize=13)
for i, ax in enumerate(axes.flat):
    if i < feat_maps.shape[-1]:
        ax.imshow(feat_maps[:, :, i], cmap="inferno")
    ax.axis("off")
plt.tight_layout()
path_fmaps = os.path.join(OUTPUT_DIR, "20_cnn_feature_maps.png")
plt.savefig(path_fmaps, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] CNN feature maps saved → {path_fmaps}")


In [ ]:
# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.05,
    fill_mode="nearest"
)
X_tr_aug = X_train_2d[:54000]    # 90% for training
y_tr_aug = y_train_oh[:54000]
X_val    = X_train_2d[54000:]
y_val    = y_train_oh[54000:]

datagen.fit(X_tr_aug)
train_gen = datagen.flow(X_tr_aug, y_tr_aug, batch_size=128, seed=SEED)


In [ ]:
# Visualise augmented samples
aug_batch_x, aug_batch_y = next(train_gen)
fig, axes = plt.subplots(2, 10, figsize=(20, 5))
fig.patch.set_facecolor("#1a1a2e")
fig.suptitle("Data Augmentation — Original vs Augmented Samples",
             color="white", fontsize=13)
for i in range(10):
    axes[0][i].imshow(X_tr_aug[i, :, :, 0], cmap="gray")
    axes[0][i].set_title("Original", color="#aaaaaa", fontsize=7)
    axes[0][i].axis("off")
    axes[1][i].imshow(aug_batch_x[i, :, :, 0], cmap="gray")
    axes[1][i].set_title("Augmented", color=SUCCESS_CLR, fontsize=7)
    axes[1][i].axis("off")
plt.tight_layout()
path_aug = os.path.join(OUTPUT_DIR, "21_augmented_samples.png")
plt.savefig(path_aug, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] Augmentation samples saved → {path_aug}")

# Advanced CNN architecture

In [ ]:
def build_advanced_cnn():
    inp = layers.Input(shape=(28, 28, 1))
 
    # Block 1
    x = layers.Conv2D(64, (3, 3), padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)
    # Block 2
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    # Block 3
    x = layers.Conv2D(256, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
 # Dense head
    x   = layers.Dense(512, activation="relu",
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.45)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.Dropout(0.30)(x)
    out = layers.Dense(10, activation="softmax")(x)
 
    model = keras.Model(inp, out, name="Advanced_CNN_Augmented")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


 

In [ ]:
adv_cnn = build_advanced_cnn()
adv_cnn.summary()
callbacks_adv = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=4, min_lr=1e-8, verbose=1),
]
steps_per_epoch = len(X_tr_aug) // 128
t0 = time.time()
hist_adv = adv_cnn.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=60,
    validation_data=(X_val, y_val),
    callbacks=callbacks_adv,
    verbose=1
)


In [ ]:
adv_time = time.time() - t0
adv_acc  = adv_cnn.evaluate(X_test_2d, y_test_oh, verbose=0)[1]
results["Advanced CNN + Augmentation"] = {
    "accuracy": adv_acc, "train_time": adv_time,
    "history": hist_adv.history, "type": "keras"
}
print(f"\n  Advanced CNN Test Accuracy: {adv_acc*100:.2f}%")
 

In [ ]:
plot_keras_history(
    hist_adv, "Advanced CNN + Augmentation",
    os.path.join(OUTPUT_DIR, "22_advcnn_history.png")
)

In [ ]:
adv_pred = np.argmax(adv_cnn.predict(X_test_2d, verbose=0), axis=1)
plot_confusion_matrix(
    confusion_matrix(y_test_raw, adv_pred),
    "Advanced CNN + Augmentation — Confusion Matrix",
    os.path.join(OUTPUT_DIR, "23_advcnn_confusion.png"), cmap="magma"
)


In [ ]:
show_misclassified(
    X_test_2d.reshape(-1, 784), y_test_raw, adv_pred,
    "Advanced CNN — Remaining Misclassified",
    os.path.join(OUTPUT_DIR, "24_advcnn_misclassified.png")
)

In [ ]:
# Per-class accuracy for best model
adv_report = classification_report(y_test_raw, adv_pred,
                                   output_dict=True)
df_adv = pd.DataFrame(adv_report).T
df_adv = df_adv[df_adv.index.isin([str(i) for i in range(10)])]
 
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")
x   = np.arange(10)
w   = 0.25
ax.bar(x - w, df_adv["precision"].astype(float), w,
       label="Precision", color="#4C72B0")
ax.bar(x,     df_adv["recall"].astype(float),    w,
       label="Recall",    color="#DD8452")
ax.bar(x + w, df_adv["f1-score"].astype(float),  w,
       label="F1-Score",  color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(range(10), color="white")
ax.set_title("Advanced CNN — Per-Class Metrics", color="white", fontsize=13)

ax.set_xlabel("Digit", color="#aaaaaa")
ax.set_ylabel("Score", color="#aaaaaa")
ax.tick_params(colors="white")
ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")

ax.set_ylim(0, 1.1)
for spine in ax.spines.values():
    spine.set_edgecolor("#444466")
plt.tight_layout()
path_adv_class = os.path.join(OUTPUT_DIR, "25_advcnn_per_class.png")
plt.savefig(path_adv_class, dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  [✓] Advanced CNN per-class metrics saved → {path_adv_class}")


# SECTION 12: FINAL COMPARISON DASHBOARD


In [ ]:
print("\n" + "=" * 70)
print("  SECTION 12 — Final Comparison Dashboard")
print("=" * 70)

In [ ]:
# Build summary DataFrame
summary = []
for name, info in results.items():
    summary.append({
        "Model": name,
        "Accuracy (%)": round(info["accuracy"] * 100, 2),
        "Train Time (s)": round(info["train_time"], 1),
        "Type": info["type"]
    })
df_summary = pd.DataFrame(summary).sort_values("Accuracy (%)")
print("\n  ── Final Model Comparison ──")
print(df_summary.to_string(index=False))
 

In [ ]:
# Mega comparison figure
fig = plt.figure(figsize=(24, 16))
fig.patch.set_facecolor("#0d0d1a")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

#   Accuracy bar chart (main)
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor("#16213e")
bar_colors = [SUCCESS_CLR if a == df_summary["Accuracy (%)"].max()
              else "#4C72B0" for a in df_summary["Accuracy (%)"]]
bars = ax1.barh(df_summary["Model"], df_summary["Accuracy (%)"],
                color=bar_colors, edgecolor="white", linewidth=0.5)
ax1.set_xlim(df_summary["Accuracy (%)"].min() - 5, 101)
ax1.set_title("Model Accuracy Comparison (%)", color="white", fontsize=14, pad=10)
ax1.set_xlabel("Test Accuracy (%)", color="#aaaaaa")
ax1.tick_params(colors="white", labelsize=9)
for spine in ax1.spines.values():
    spine.set_edgecolor("#444466")
for bar, acc in zip(bars, df_summary["Accuracy (%)"]):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
             f"{acc:.2f}%", va="center", ha="left",
             color="white", fontsize=10, fontweight="bold")
#  Training time bar chart
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor("#16213e")
ax2.barh(df_summary["Model"], df_summary["Train Time (s)"],
         color=sns.color_palette("rocket", len(df_summary)),
         edgecolor="white", linewidth=0.4)
ax2.set_title("Training Time (seconds)", color="white", fontsize=12, pad=10)
ax2.set_xlabel("Seconds", color="#aaaaaa")
ax2.tick_params(colors="white", labelsize=8)
for spine in ax2.spines.values():
    spine.set_edgecolor("#444466")


 
# Keras models: validation accuracy over time
ax3 = fig.add_subplot(gs[1, :2])
ax3.set_facecolor("#16213e")
keras_palette = sns.color_palette("tab10", 4)
kmodels = [(n, v) for n, v in results.items() if v["type"] == "keras"]
for (name, info), col in zip(kmodels, keras_palette):
    val_acc = [a * 100 for a in info["history"]["val_accuracy"]]
    ax3.plot(val_acc, label=name, linewidth=2.5, color=col)
ax3.set_title("Keras Models — Validation Accuracy Over Epochs",
              color="white", fontsize=13)
ax3.set_xlabel("Epoch", color="#aaaaaa")
ax3.set_ylabel("Val Accuracy (%)", color="#aaaaaa")
ax3.tick_params(colors="white")
ax3.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white",
           fontsize=9, loc="lower right")
for spine in ax3.spines.values():
    spine.set_edgecolor("#444466")
 
# Accuracy improvement step plot
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor("#16213e")
model_names_sorted = df_summary["Model"].tolist()
accs_sorted        = df_summary["Accuracy (%)"].tolist()
ax4.step(range(len(model_names_sorted)), accs_sorted,
         where="mid", color=SUCCESS_CLR, linewidth=2.5)
ax4.scatter(range(len(model_names_sorted)), accs_sorted,
            color=SUCCESS_CLR, zorder=5, s=80)
ax4.set_xticks(range(len(model_names_sorted)))
ax4.set_xticklabels([m[:12] + "…" if len(m) > 12 else m
                     for m in model_names_sorted],
                    rotation=40, ha="right", color="white", fontsize=7)
ax4.set_title("Accuracy Improvement Journey", color="white", fontsize=11)


